# Attention-Weight & Attention-Pattern Analysis — 4 models

Companion to `embedding_analysis.ipynb`, working through
`weights_analysis/todo_attention.txt` for four checkpoints:

| Key | Directory | Role |
|-----|-----------|------|
| `Llama-3-8B`        | `llama-3-8b`                   | original Meta base model |
| `SinLlama_v01`      | `SinLlama_v01`                | Sinhala model from Llama-3 (CPT) — **parent of cpt & instruct** |
| `SinLlama_cpt`      | `SinLlama_cpt`                 | branched from v01 |
| `SinLlama_Instruct` | `SinLlama_Backtrianx_Instruct`| branched from v01 (Bactrian-X instruct) |

**Lineage:** `Llama-3-8B → SinLlama_v01 → {SinLlama_cpt, SinLlama_Instruct}` —
`cpt` and `instruct` are **parallel branches off v01**, not sequential to each
other. Comparisons follow this order.

**All four share the identical attention architecture** (only the embedding /
LM-head vocab differs), so every attention weight is comparable per
`(layer, head)` — we can watch how each training stage reshaped attention.

### Architecture (`config.json`)
* 32 layers, `hidden = 4096`
* **32 query heads, 8 key/value heads** (GQA), `head_dim = 128` (4 Q heads / KV head)

### Two regimes
* **Part A — static / weight-space** (§1,2,5,6,7,8,11,14,16): reads only
  `q/k/v/o_proj` (~2.7 GB / model) via `safetensors`. Runs anywhere.
* **Part B — dynamic / activation-space** (§3,4,9,10,12,13,15,17): needs a real
  forward with `output_attentions=True`, which **requires the full 8B model and
  `attn_implementation="eager"`** (flash/sdpa don't return attention weights).
  Auto-selects GPU if it fits, else CPU.

> **Memory:** Part A holds four models' projections ≈ 21 GB float32 RAM; Part B
> loads each full 8B model one at a time (~16 GB) and frees it. Comfortable on
> the MI300X pod; trim `MODEL_DIRS` / `DYN_MODELS` on smaller boxes.

## 0. Setup

In [ ]:
import os, re, json, math, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from safetensors import safe_open
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import linkage, dendrogram
import umap

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

# --- Portable model paths (laptop AND pod; case-insensitive dir match) ----
CANDIDATE_ROOTS = [
    "/ml/SinLlama_CPT/models",
    os.path.expanduser("~/sinllama-continual-pretraining/models"),
    "/root/sinllama-continual-pretraining/models",
    "./models",
]
MODELS_ROOT = next((r for r in CANDIDATE_ROOTS if os.path.isdir(r)), CANDIDATE_ROOTS[0])
# Order follows the model lineage: llama -> v01 -> {cpt, instruct}.
MODEL_DIRS = {
    "Llama-3-8B":        ["llama-3-8b"],
    "SinLlama_v01":      ["SinLlama_v01"],
    "SinLlama_cpt":      ["SinLlama_cpt"],
    "SinLlama_Instruct": ["SinLlama_Backtrianx_Instruct", "SinLlama_Backtrianx_instruct"],
}
def resolve_dir(root, candidates):
    existing = {d.lower(): d for d in os.listdir(root)
                if os.path.isdir(os.path.join(root, d))}
    for c in candidates:
        if c.lower() in existing:
            return os.path.join(root, existing[c.lower()])
    raise FileNotFoundError(f"none of {candidates} under {root}")
MODEL_PATHS = {k: resolve_dir(MODELS_ROOT, v) for k, v in MODEL_DIRS.items()}
MODEL_KEYS = list(MODEL_PATHS)          # canonical iteration order
REF = MODEL_KEYS[0]                     # base model for single-model reference plots

# Architecture constants (identical for every model).
N_LAYERS, HIDDEN = 32, 4096
N_HEADS, N_KV_HEADS, HEAD_DIM = 32, 8, 128
GROUP = N_HEADS // N_KV_HEADS

FIG_DIR = os.path.join(os.path.dirname(MODELS_ROOT.rstrip("/")),
                       "weights_analysis", "figures_attention")
if not os.path.isdir(os.path.dirname(FIG_DIR)):
    FIG_DIR = "figures_attention"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, name), bbox_inches="tight"); plt.show()

DEPTH_CMAP = plt.cm.viridis

# --- Singular values for Part A -------------------------------------------
# NOTE: benchmarked on the MI300X, a *single* dense 4096^2 SVD is faster on the
# multithreaded CPU LAPACK than on the GPU (rocSOLVER svdvals ~0.5x). The GPU
# only wins for SVD when many matrices are batched into one call. So SVDs stay
# on the CPU, but each one is computed ONCE (cached in SV, reused by A8).
def svdvals(mat):
    "Singular values (descending) of a 2-D matrix."
    return np.linalg.svd(mat, compute_uv=False)

print("MODELS_ROOT:", MODELS_ROOT, "| figures ->", FIG_DIR)
for k, p in MODEL_PATHS.items():
    print(f"  {k:20s} -> {p}")

# PART A — Static / weight-space analysis

Reads only the attention projection matrices; no full model instantiation.

## A1 · §1 Load & inspect attention weights

Pull every `q/k/v/o_proj.weight` across all 32 layers from the safetensors
shards (each shard opened once), for all four models.

In [ ]:
def load_attn_weights(model_path):
    "Return {(layer, proj): float32 array} for proj in q/k/v/o, plus orig dtype."
    wmap = json.load(open(os.path.join(model_path,
                    "model.safetensors.index.json")))["weight_map"]
    want = {f"model.layers.{L}.self_attn.{p}_proj.weight": (L, p)
            for L in range(N_LAYERS) for p in ("q", "k", "v", "o")}
    by_shard = {}
    for name in want:
        by_shard.setdefault(wmap[name], []).append(name)
    W, dtype = {}, None
    for shard, names in by_shard.items():
        with safe_open(os.path.join(model_path, shard), framework="pt",
                       device="cpu") as f:
            for name in names:
                t = f.get_tensor(name); dtype = t.dtype
                W[want[name]] = t.to(torch.float32).numpy()
    return W, dtype

ATTN = {}                                   # model_key -> {(layer, proj): array}
for key in MODEL_KEYS:
    W, dtype = load_attn_weights(MODEL_PATHS[key])
    ATTN[key] = W
    q, k, v, o = W[(0,"q")], W[(0,"k")], W[(0,"v")], W[(0,"o")]
    n_params = sum(a.size for a in W.values())
    print(f"{key:20s} dtype={dtype} | q{q.shape} k{k.shape} v{v.shape} o{o.shape} "
          f"| attn params={n_params:,} ({n_params*2/1e9:.2f} GB bf16)")

SUMMARY = {k: {} for k in MODEL_KEYS}
HEADFEAT, COPYSCORE = {}, {}                 # derived arrays (kept out of ATTN)

def q_head(W, L, h):  return W[(L,"q")][h*HEAD_DIM:(h+1)*HEAD_DIM]
def k_head(W, L, h):  return W[(L,"k")][h*HEAD_DIM:(h+1)*HEAD_DIM]
def v_head(W, L, h):  return W[(L,"v")][h*HEAD_DIM:(h+1)*HEAD_DIM]
def o_head(W, L, h):  return W[(L,"o")][:, h*HEAD_DIM:(h+1)*HEAD_DIM]

## A1b · Per-layer weight drift from base

Cosine similarity between the base model's projection and each SinLlama stage's
projection, per layer — how far continual pretraining moved each attention
matrix (1 = unchanged). A direct 4-model training-progression signal.

In [ ]:
def layer_drift(base_W, var_W, proj):
    "Per-layer cosine between flattened projection matrices."
    out = []
    for L in range(N_LAYERS):
        a = base_W[(L, proj)].ravel(); b = var_W[(L, proj)].ravel()
        out.append(float(a @ b / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-8)))
    return np.array(out)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, p in zip(axes, ("q", "k", "v", "o")):
    for key in MODEL_KEYS:
        if key == REF:
            continue
        dr = layer_drift(ATTN[REF], ATTN[key], p)
        ax.plot(dr, marker="o", ms=3, label=key)
        SUMMARY[key][f"mean_drift_{p}"] = float(dr.mean())
    ax.set_title(f"{p}_proj cosine to base"); ax.set_xlabel("layer")
axes[0].set_ylabel("cos(base, variant)"); axes[0].legend(fontsize=8)
savefig("A1b_weight_drift.png")

## A2 · §2 Basic statistics & Frobenius norms

In [ ]:
rows = []
for key in MODEL_KEYS:
    for L in range(N_LAYERS):
        for p in ("q", "k", "v", "o"):
            m = ATTN[key][(L, p)]
            rows.append({"model": key, "layer": L, "proj": p,
                         "fro": np.linalg.norm(m), "std": m.std(),
                         "absmax": np.abs(m).max()})
norm_df = pd.DataFrame(rows)
display(norm_df.groupby(["model", "proj"])[["fro", "std", "absmax"]].mean().round(4))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharex=True)
for ax, p in zip(axes, ("q", "k", "v", "o")):
    for key in MODEL_KEYS:
        sub = norm_df[(norm_df.model == key) & (norm_df.proj == p)]
        ax.plot(sub.layer, sub.fro, marker="o", ms=3, label=key)
    ax.set_title(f"{p}_proj Frobenius norm"); ax.set_xlabel("layer")
axes[0].set_ylabel("||W||_F"); axes[0].legend(fontsize=8)
savefig("A2_frobenius_by_layer.png")

plt.figure(figsize=(8, 4.5))
for key in MODEL_KEYS:
    vals = np.concatenate([ATTN[key][(L,"q")].ravel()[::50] for L in range(0, N_LAYERS, 4)])
    plt.hist(vals, bins=200, density=True, alpha=0.5, label=key)
plt.title("q_proj weight-magnitude distribution (sampled)")
plt.xlabel("weight value"); plt.ylabel("density"); plt.legend(fontsize=8)
savefig("A2_magnitude_dist.png")

for key in MODEL_KEYS:
    g = norm_df[norm_df.model == key].groupby("layer")["fro"].mean()
    z = (g - g.mean()) / g.std()
    print(f"{key}: outlier layers (|z(mean Fro)|>2): {list(z[np.abs(z) > 2].index)}")

## A3 · §8 SVD / spectral analysis (effective rank)

All q/k/v/o singular values are computed **once** and cached in `SV`, reused by
A8 (the earlier version recomputed every SVD twice).

In [ ]:
def effective_rank(sv):
    p = sv / sv.sum(); p = p[p > 0]
    return float(np.exp(-(p * np.log(p)).sum()))

# Singular values for every (model, layer, proj), computed once on the GPU.
SV = {}
erank = {key: {"q": [], "o": []} for key in MODEL_KEYS}
spectra = {}
for key in MODEL_KEYS:
    for L in range(N_LAYERS):
        for p in ("q", "k", "v", "o"):
            SV[(key, L, p)] = svdvals(ATTN[key][(L, p)])
        for p in ("q", "o"):
            erank[key][p].append(effective_rank(SV[(key, L, p)]))
            if L in (0, N_LAYERS//2, N_LAYERS-1):
                spectra[(key, L, p)] = SV[(key, L, p)]
    SUMMARY[key]["q_effrank_mean"] = float(np.mean(erank[key]["q"]))
    SUMMARY[key]["o_effrank_mean"] = float(np.mean(erank[key]["o"]))
    print(f"{key}: q eff-rank mean={np.mean(erank[key]['q']):.1f} | "
          f"o eff-rank mean={np.mean(erank[key]['o']):.1f} (of {HIDDEN})")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for (key, L, p), sv in spectra.items():
    if p == "q" and L == N_LAYERS//2:
        ax[0].semilogy(sv, alpha=0.8, label=f"{key} L{L}")
ax[0].set_title("q_proj singular spectra (mid layer, log)")
ax[0].set_xlabel("index"); ax[0].set_ylabel("singular value"); ax[0].legend(fontsize=8)
for key in MODEL_KEYS:
    ax[1].plot(erank[key]["q"], marker="o", ms=3, label=key)
ax[1].set_title("q_proj effective rank vs depth")
ax[1].set_xlabel("layer"); ax[1].set_ylabel("effective rank"); ax[1].legend(fontsize=8)
savefig("A3_svd_effrank.png")

## A4 · §5 Cosine similarity across heads (redundant heads)

Within each layer, pairwise cosine between flattened query-head `q_proj` slices;
high off-diagonal = redundant heads. Heatmaps for the reference model + the
most-redundant pair per layer for every model.

In [ ]:
def head_cos_matrix(W, L, proj="q", n=N_HEADS):
    slicer = {"q": q_head, "k": k_head, "v": v_head}[proj]
    vecs = np.stack([slicer(W, L, h).ravel() for h in range(n)])
    vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs @ vecs.T

SHOW_LAYERS = [0, N_LAYERS//2, N_LAYERS-1]
fig, axes = plt.subplots(1, len(SHOW_LAYERS), figsize=(5*len(SHOW_LAYERS), 4.5))
for c, L in enumerate(SHOW_LAYERS):
    S = head_cos_matrix(ATTN[REF], L, "q")
    sns.heatmap(S, ax=axes[c], cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True,
                cbar=(c == len(SHOW_LAYERS)-1))
    axes[c].set_title(f"{REF} L{L} q-head cosine")
savefig("A4_head_cosine.png")

for key in MODEL_KEYS:
    worst = []
    for L in range(N_LAYERS):
        S = head_cos_matrix(ATTN[key], L, "q"); np.fill_diagonal(S, -1)
        i, j = np.unravel_index(S.argmax(), S.shape); worst.append((L, i, j, S[i, j]))
    top = sorted(worst, key=lambda x: -x[3])[:5]
    print(f"{key}: most-redundant q-head pairs: "
          + ", ".join(f"(L{L},{i},{j},{c:.2f})" for L, i, j, c in top))

## A5 · §6 + §7 Head embeddings → PCA / UMAP + clustering

Each of the 1024 query heads is represented by a compact spectral signature
(top singular values + norm of its `q_proj` slice); PCA/UMAP colour heads by
layer, K-Means groups them.

In [ ]:
def head_signature(W, L, h, topk=16):
    sv = svdvals(q_head(W, L, h))[:topk]
    return np.concatenate([sv, [np.linalg.norm(q_head(W, L, h))]])

for key in MODEL_KEYS:
    feats, layer_of = [], []
    for L in range(N_LAYERS):
        for h in range(N_HEADS):
            feats.append(head_signature(ATTN[key], L, h)); layer_of.append(L)
    feats = np.asarray(feats); layer_of = np.asarray(layer_of)
    feats = (feats - feats.mean(0)) / (feats.std(0) + 1e-8)

    pca = PCA(n_components=2, random_state=SEED).fit_transform(feats)
    um  = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(feats)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for coords, a, name in [(pca, ax[0], "PCA"), (um, ax[1], "UMAP")]:
        sc = a.scatter(coords[:, 0], coords[:, 1], c=layer_of, cmap="viridis", s=18)
        a.set_title(f"{key} — {name} of 1024 heads (colour = layer)")
    fig.colorbar(sc, ax=ax[1], label="layer depth")
    savefig(f"A5_head_pca_umap_{key}.png")

    km = KMeans(n_clusters=8, n_init=10, random_state=SEED).fit(feats)
    HEADFEAT[key] = (feats, layer_of, km.labels_)
    print(f"{key}: head clusters concentrate at depths -> "
          + str({c: int(np.median(layer_of[km.labels_ == c])) for c in range(8)}))

In [ ]:
# Ward dendrogram of heads for the reference model.
feats, layer_of, _ = HEADFEAT[REF]
Z = linkage(feats, method="ward")
plt.figure(figsize=(12, 4))
dendrogram(Z, no_labels=True, color_threshold=0.7 * Z[:, 2].max())
plt.title(f"Ward hierarchical clustering of 1024 heads — {REF}")
plt.ylabel("merge distance")
savefig("A5_head_dendrogram.png")

## A6 · §14 (weight-space) All-head signature-similarity heatmap

In [ ]:
feats, layer_of, _ = HEADFEAT[REF]
unit = feats / np.linalg.norm(feats, axis=1, keepdims=True)
plt.figure(figsize=(7.5, 6.5))
sns.heatmap(unit @ unit.T, cmap="magma", square=True,
            cbar_kws={"label": "cosine (signature)"})
plt.title(f"All-head signature similarity (1024x1024) — {REF}")
plt.xlabel("head index (layer-major)"); plt.ylabel("head index")
savefig("A6_all_head_similarity.png")

## A7 · §16 (weight-space) OV-circuit copying proxy

`trace(W_O_head @ W_V_head) / ||·||_F` per head — high positive = copying /
induction-like (Transformer-Circuits framework). One heatmap per model.

In [ ]:
def ov_copying_proxy(W, L, h):
    kvh = h // GROUP                       # GQA: query head -> shared KV head
    Wo, Wv = o_head(W, L, h), v_head(W, L, kvh)
    return np.einsum("ij,ji->", Wo, Wv) / (np.linalg.norm(Wo)*np.linalg.norm(Wv) + 1e-8)

fig, axes = plt.subplots(1, len(MODEL_KEYS), figsize=(5.5*len(MODEL_KEYS), 4.5))
axes = np.atleast_1d(axes)
for ax, key in zip(axes, MODEL_KEYS):
    copy = np.array([[ov_copying_proxy(ATTN[key], L, h) for h in range(N_HEADS)]
                     for L in range(N_LAYERS)])
    COPYSCORE[key] = copy
    SUMMARY[key]["max_copying_proxy"] = float(copy.max())
    sns.heatmap(copy, ax=ax, cmap="RdBu_r", center=0, cbar_kws={"label": "OV copy proxy"})
    ax.set_title(key); ax.set_xlabel("head"); ax.set_ylabel("layer")
savefig("A7_ov_copying.png")
for key in MODEL_KEYS:
    copy = COPYSCORE[key]
    top = np.dstack(np.unravel_index(np.argsort(copy.ravel())[::-1][:5], copy.shape))[0]
    print(f"{key}: strongest copying heads (L,h): " + ", ".join(f"(L{L},{h})" for L, h in top))

## A8 · §11 (weight-space) Q/K/V anisotropy summary

In [ ]:
rows = []
for key in MODEL_KEYS:
    for p in ("q", "k", "v", "o"):
        shares = [SV[(key, L, p)][0] / SV[(key, L, p)].sum() for L in range(N_LAYERS)]
        rows.append({"model": key, "proj": p, "top1_sv_share": np.mean(shares)})
display(pd.DataFrame(rows).pivot(index="proj", columns="model",
                                 values="top1_sv_share").round(4))
print("Higher top-1 singular share => more anisotropic projection.")

In [ ]:
del ATTN; gc.collect()
print("Released Part-A weight arrays. Ready for Part B.")

# PART B — Dynamic / activation-space analysis

Real forward passes with `attn_implementation="eager"`. Auto-selects GPU if it
fits, else CPU.

## B0 · Device guard, model loader & sample inputs

In [ ]:
def pick_device(param_billion=8.03, dtype_bytes=2, overhead=1.4):
    if torch.cuda.is_available():
        free, _ = torch.cuda.mem_get_info()
        if free > param_billion * 1e9 * dtype_bytes * overhead:
            return "cuda"
    return "cpu"

DEVICE = pick_device()
DTYPE = torch.bfloat16
MAX_LEN = 48 if DEVICE == "cpu" else 96
print(f"Part B device = {DEVICE} | dtype = {DTYPE} | MAX_LEN = {MAX_LEN}")
if DEVICE == "cpu":
    print("NOTE: 8B forward on CPU is slow. On the MI300X this runs on GPU.")

# Which models to analyse dynamically (loaded one at a time and freed).
DYN_MODELS = list(MODEL_KEYS)      # e.g. [MODEL_KEYS[0], "SinLlama_Instruct"] to trim

SAMPLE_TEXTS = [
    "The quick brown fox jumps over the lazy dog near the river bank.",
    "In 2024, sales rose by 15% to $3.2 million, up from 2.8 last year.",
    "She said, \"Come here!\" and then walked away without another word.",
    "ශ්‍රී ලංකාව දකුණු ආසියාවේ පිහිටි දිවයිනකි. එහි අගනුවර කොළඹ නගරයයි.",
]

@torch.no_grad()
def capture_attentions(model, tok, text):
    enc = tok(text, return_tensors="pt", truncation=True,
              max_length=MAX_LEN).to(model.device)
    out = model(**enc, output_attentions=True)
    return [a[0].float().cpu().numpy() for a in out.attentions], enc["input_ids"][0].cpu().numpy()

def load_full_model(path):
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(
        path, torch_dtype=DTYPE, attn_implementation="eager", low_cpu_mem_usage=True)
    return model.to(DEVICE).eval(), tok

DYN = {}

## B1 · §3 + §17 Attention entropy & sparsity

Per `(layer, head)`, averaged over sample texts and query positions: entropy
(sharp vs diffuse), top-1 mass (sparsity), attention-sink mass on position 0,
and mean signed distance `sum_j p_ij (i-j)` (local vs long-range).

In [ ]:
def per_head_metrics(model, tok, texts):
    ent = np.zeros((N_LAYERS, N_HEADS)); top1 = np.zeros_like(ent)
    sink = np.zeros_like(ent);           dist = np.zeros_like(ent); n = 0
    for t in texts:
        atts, ids = capture_attentions(model, tok, t)
        S = len(ids)
        dmat = (np.arange(S)[:, None] - np.arange(S)[None, :]).astype(np.float32)
        for L in range(N_LAYERS):
            A = atts[L]
            logA = np.where(A > 0, np.log(A + 1e-12), 0.0)
            r = slice(1, S)
            ent[L]  += (-(A*logA).sum(-1))[:, r].mean(1)
            top1[L] += A.max(-1)[:, r].mean(1)
            sink[L] += A[:, r, 0].mean(1)
            dist[L] += (A*dmat[None])[:, r].sum(-1).mean(1)
        n += 1
    return {k: v/n for k, v in dict(entropy=ent, top1=top1, sink=sink, distance=dist).items()}

@torch.no_grad()
def induction_scores(model):
    n = 24
    vocab = model.config.vocab_size
    seq = rng.integers(5, vocab - 1000, size=n)
    ids = np.concatenate([[model.config.bos_token_id or 1], seq, seq])
    out = model(torch.tensor(ids[None], device=model.device), output_attentions=True)
    S = len(ids); score = np.zeros((N_LAYERS, N_HEADS))
    for qi in range(1 + n, S):
        target = 1 + np.where(seq == ids[qi])[0][0] + 1
        for L in range(N_LAYERS):
            score[L] += out.attentions[L][0, :, qi, target].float().cpu().numpy()
    return score / n

# Load each model ONCE, extract EVERY per-model array (entropy metrics + the
# induction map used by B2), then FREE non-reference models immediately — only
# the reference model stays resident, for the reference-only cells below.
# Holding all four 8B models at once exhausts GPU memory (especially with the
# three notebooks running in parallel); that previously aborted this loop after
# the first model, leaving DYN with one model and the 4-model plots blank.
for key in DYN_MODELS:
    print(f"\n>>> loading {key} on {DEVICE} ...")
    model, tok = load_full_model(MODEL_PATHS[key])
    DYN[key] = {}
    DYN[key].update(per_head_metrics(model, tok, SAMPLE_TEXTS))
    DYN[key]["induction"] = induction_scores(model)
    SUMMARY[key]["mean_entropy"] = float(DYN[key]["entropy"].mean())
    print(f"{key}: mean entropy = {DYN[key]['entropy'].mean():.3f} nats | "
          f"mean distance = {DYN[key]['distance'].mean():.2f} tokens")
    if key == REF:
        DYN[key]["_model"] = model; DYN[key]["_tok"] = tok
    else:
        del model, tok; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

fig, axes = plt.subplots(1, len(DYN_MODELS), figsize=(6*len(DYN_MODELS), 4.6))
axes = np.atleast_1d(axes)
for ax, key in zip(axes, DYN_MODELS):
    sns.heatmap(DYN[key]["entropy"], ax=ax, cmap="viridis",
                cbar_kws={"label": "entropy (nats)"})
    ax.set_title(key); ax.set_xlabel("head"); ax.set_ylabel("layer")
savefig("B1_entropy_heatmap.png")

## B2 · §4 Head-behaviour classification (incl. induction)

Local / global / sink / induction heads. Induction is measured with a crafted
`[BOS, r_1..r_n, r_1..r_n]` repeated random sequence — an induction head attends
from a token to the position right after its previous occurrence.

In [ ]:
for key in DYN_MODELS:
    d = DYN[key]; ind = d["induction"]          # computed in B1 while the model was live
    local = d["distance"] < np.percentile(d["distance"], 33)
    globl = d["distance"] > np.percentile(d["distance"], 90)
    sinky = d["sink"] > 0.5; induct = ind > 0.2
    SUMMARY[key]["n_induction_heads"] = int(induct.sum())
    SUMMARY[key]["n_sink_heads"] = int(sinky.sum())
    top = np.dstack(np.unravel_index(np.argsort(ind.ravel())[::-1][:5], ind.shape))[0]
    print(f"{key}: local={local.sum()} global={globl.sum()} sink={sinky.sum()} "
          f"induction={induct.sum()} | top induction: "
          + ", ".join(f"(L{L},{h})={ind[L,h]:.2f}" for L, h in top))

## B3 · §15 Positional bias / attention-distance decay (reference model)

In [ ]:
def distance_decay(model, tok, texts, max_d=40):
    acc = np.zeros((N_LAYERS, max_d)); cnt = np.zeros(max_d)
    for t in texts:
        atts, ids = capture_attentions(model, tok, t)
        S = len(ids)
        for i in range(1, S):
            for j in range(i + 1):
                dd = i - j
                if dd < max_d:
                    for L in range(N_LAYERS):
                        acc[L, dd] += atts[L][:, i, j].mean()
                    cnt[dd] += 1
    return acc / np.maximum(cnt, 1)

decay = distance_decay(DYN[REF]["_model"], DYN[REF]["_tok"], SAMPLE_TEXTS)
plt.figure(figsize=(9, 5))
for L in range(0, N_LAYERS, 4):
    plt.plot(decay[L], color=DEPTH_CMAP(L / N_LAYERS), label=f"L{L}")
plt.yscale("log"); plt.xlabel("query-key distance (i-j)")
plt.ylabel("mean attention probability")
plt.title(f"Attention distance-decay by layer — {REF}"); plt.legend(fontsize=8, ncol=2)
savefig("B3_distance_decay.png")

## B4 · §9 + §18 Attention-map visualisation (reference model)

In [ ]:
atts, ids = capture_attentions(DYN[REF]["_model"], DYN[REF]["_tok"], SAMPLE_TEXTS[0])
labels = DYN[REF]["_tok"].convert_ids_to_tokens(ids)
picks = [(0, 0), (N_LAYERS//2, 5), (N_LAYERS-1, 0), (N_LAYERS-1, 15)]
fig, axes = plt.subplots(1, len(picks), figsize=(5*len(picks), 4.6))
for ax, (L, h) in zip(axes, picks):
    sns.heatmap(atts[L][h], ax=ax, cmap="viridis", cbar=False, square=True,
                xticklabels=labels, yticklabels=labels)
    ax.set_title(f"{REF} L{L} H{h}"); ax.tick_params(labelsize=5)
savefig("B4_attention_maps.png")

## B5 · §12 Layer-wise evolution (all dynamic models overlaid)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
for key in DYN_MODELS:
    d = DYN[key]
    ax[0].plot(d["entropy"].mean(1),  marker="o", ms=3, label=key)
    ax[1].plot(d["distance"].mean(1), marker="o", ms=3, label=key)
    ax[2].plot(d["entropy"].std(1),   marker="o", ms=3, label=key)
ax[0].set_title("mean entropy vs depth")
ax[1].set_title("mean attention distance vs depth")
ax[2].set_title("entropy spread across heads (specialisation)")
for a in ax: a.set_xlabel("layer"); a.legend(fontsize=8)
savefig("B5_layerwise_evolution.png")

## B6 · §13 Token-conditioned attention (reference model)

In [ ]:
STOP = {"the","a","an","of","to","in","and","is","was","that","it","for","on","with"}
def token_type(tok, tid):
    s = tok.decode([int(tid)]).strip()
    if tid in set(tok.all_special_ids): return "special"
    if s == "": return "whitespace"
    if re.search(r"[඀-෿]", s): return "sinhala"
    if s.isdigit(): return "number"
    if all(not c.isalnum() for c in s): return "punctuation"
    if s.lower() in STOP: return "stopword"
    return "english"

model, tok = DYN[REF]["_model"], DYN[REF]["_tok"]
recv = {}
for t in SAMPLE_TEXTS:
    atts, ids = capture_attentions(model, tok, t)
    types = [token_type(tok, i) for i in ids]
    col_mass = np.mean([a.mean(0).mean(0) for a in atts], axis=0)
    for j, ty in enumerate(types):
        recv.setdefault(ty, []).append(col_mass[j])
summary = {k: float(np.mean(v)) for k, v in recv.items()}
order = sorted(summary, key=summary.get, reverse=True)
plt.figure(figsize=(8, 4.2))
sns.barplot(x=order, y=[summary[k] for k in order])
plt.ylabel("mean received-attention fraction")
plt.title(f"Attention received by token type — {REF}"); plt.xticks(rotation=25)
savefig("B6_token_conditioned.png")
print("received-attention by type:", {k: round(v, 4) for k, v in summary.items()})

## B7 · §14 Attention-map similarity across heads (reference model)

In [ ]:
atts, ids = capture_attentions(DYN[REF]["_model"], DYN[REF]["_tok"], SAMPLE_TEXTS[0])
L_probe = N_LAYERS // 2
flat = atts[L_probe].reshape(N_HEADS, -1)
flat /= (np.linalg.norm(flat, axis=1, keepdims=True) + 1e-8)
sim = flat @ flat.T
plt.figure(figsize=(6.5, 5.5))
sns.heatmap(sim, cmap="magma", square=True, cbar_kws={"label": "cosine"})
plt.title(f"Attention-map similarity across heads — {REF} L{L_probe}")
plt.xlabel("head"); plt.ylabel("head")
savefig("B7_attn_map_similarity.png")
off = sim - np.eye(N_HEADS); i, j = np.unravel_index(off.argmax(), off.shape)
print(f"most similar head pair in L{L_probe}: H{i} ~ H{j}  cos={off[i,j]:.3f}")

## B8 · §10 Head-importance via ablation (reference model, configurable)

Zero each head's `o_proj` columns and measure the LM-loss increase.
`n_layers × n_heads` forwards — trivial on the MI300X (set
`ABLATION_LAYERS = range(N_LAYERS)`), slow on CPU (default a few layers).

In [ ]:
ABLATION_LAYERS = list(range(N_LAYERS)) if DEVICE == "cuda" else [0, N_LAYERS//2, N_LAYERS-1]
ABLATION_TEXT = "The capital of France is Paris and the capital of Japan is Tokyo."

@torch.no_grad()
def head_importance(model, tok, layers):
    ids = tok(ABLATION_TEXT, return_tensors="pt").to(model.device)["input_ids"]
    base = model(ids, labels=ids).loss.item()
    imp = np.full((N_LAYERS, N_HEADS), np.nan)
    for L in layers:
        oproj = model.model.layers[L].self_attn.o_proj
        for h in range(N_HEADS):
            sl = slice(h*HEAD_DIM, (h+1)*HEAD_DIM)
            def hook(mod, args, sl=sl):
                x = args[0].clone(); x[..., sl] = 0
                return (x,) + args[1:]
            handle = oproj.register_forward_pre_hook(hook)
            imp[L, h] = model(ids, labels=ids).loss.item() - base
            handle.remove()
    return base, imp

print(f"Ablating {len(ABLATION_LAYERS)*N_HEADS} heads on {DEVICE} ...")
base, imp = head_importance(DYN[REF]["_model"], DYN[REF]["_tok"], ABLATION_LAYERS)
plt.figure(figsize=(9, 4))
sns.heatmap(imp, cmap="rocket_r", cbar_kws={"label": "loss increase"}, mask=np.isnan(imp))
plt.title(f"Head importance (Δloss when ablated) — {REF} (base={base:.3f})")
plt.xlabel("head"); plt.ylabel("layer")
savefig("B8_head_importance.png")
flat = [(L, h, imp[L, h]) for L in ABLATION_LAYERS for h in range(N_HEADS)]
print("most important heads (L,h,Δloss):",
      [(L, h, round(v, 4)) for L, h, v in sorted(flat, key=lambda x: -x[2])[:8]])

## B9 · §16 Attention rollout (reference model)

In [ ]:
def attention_rollout(atts):
    S = atts[0].shape[-1]; R = np.eye(S)
    for A in atts:
        Am = 0.5 * A.mean(0) + 0.5 * np.eye(S); Am /= Am.sum(-1, keepdims=True)
        R = Am @ R
    return R

atts, ids = capture_attentions(DYN[REF]["_model"], DYN[REF]["_tok"], SAMPLE_TEXTS[0])
R = attention_rollout(atts)
labels = DYN[REF]["_tok"].convert_ids_to_tokens(ids)
plt.figure(figsize=(7.5, 6))
sns.heatmap(R, cmap="viridis", xticklabels=labels, yticklabels=labels, square=True)
plt.title(f"Attention rollout (info flow) — {REF}"); plt.tick_params(labelsize=5)
savefig("B9_rollout.png")
print(f"{REF}: final-token rollout mass on -> {[labels[j] for j in np.argsort(-R[-1])[:5]]}")

In [ ]:
for key in DYN_MODELS:
    DYN[key].pop("_model", None); DYN[key].pop("_tok", None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released full models.")

# PART C — §19 Synthesis: research questions

In [ ]:
summary_df = pd.DataFrame(SUMMARY).T
display(summary_df.round(3))

def g(key, m, d=float("nan")):
    return SUMMARY.get(key, {}).get(m, d)

print("\n================ ANSWERS ================")
for key in MODEL_KEYS:
    print(f"[{key}]  induction heads={g(key,'n_induction_heads')}, "
          f"sink heads={g(key,'n_sink_heads')}, mean entropy={g(key,'mean_entropy'):.3f}, "
          f"q eff-rank={g(key,'q_effrank_mean'):.0f}/{HIDDEN}, "
          f"max copy proxy={g(key,'max_copying_proxy'):.2f}, "
          f"weight drift q={g(key,'mean_drift_q'):.3f}")
print("\n• Weight drift (A1b) is the cleanest training-stage signal across the 4 models.")
print("• Local-with-depth, long-range layers -> B5 distance curves.")
print("• Redundant / unique heads -> A4, A6, B7.")

### Reading the 4-model comparison

* **Part A weight-space** cleanly separates the branches: per-layer drift (A1b,
  measured against base), Frobenius norms (A2), effective ranks (A3/A8) and OV
  copying maps (A7) show *where* `Llama-3 → v01 → {cpt, instruct}` changed
  attention most — compare the cpt and instruct siblings against their parent v01.
* **Part B activation-space** depends on the input language — the Sinhala sample
  exercises the SinLlama models' adapted heads; compare entropy heatmaps (B1),
  induction counts (B2) and layer-wise evolution (B5).
* On the MI300X, widen `ABLATION_LAYERS` to `range(N_LAYERS)` and add longer /
  more `SAMPLE_TEXTS` for publication-grade Part-B results.